# 05 — Churn Definition
Churn is not directly observed in a non-subscription marketplace, so the inactivity threshold is derived from actual repeat-purchase gaps.


In [ ]:
customer_orders = customer_orders.sort_values(["customer_unique_id","order_purchase_timestamp"])
customer_orders["previous_purchase"] = customer_orders.groupby("customer_unique_id")["order_purchase_timestamp"].shift(1)
customer_orders["purchase_gap_days"] = (customer_orders["order_purchase_timestamp"]-customer_orders["previous_purchase"]).dt.days
purchase_gaps = customer_orders["purchase_gap_days"].dropna()


## Observed gap distribution
Count 3,345; median 28 days; 75th percentile 119; 80th 151; 90th 241; 95th 312; maximum 608 days.


In [ ]:
for t in [60,90,120,180]:
    pct=(purchase_gaps<=t).mean()*100
    print(f"{t} days: {pct:.2f}%")


Observed coverage: 60d **62.87%**, 90d **69.63%**, 120d **75.43%**, 180d **83.86%**.
**Working threshold:** 180 days. It is not final until we validate on behavioral gaps (>7 days) and define leakage-safe observation/prediction windows.


In [ ]:
behavioral_gaps = purchase_gaps[purchase_gaps > 7]
behavioral_gaps.describe(percentiles=[0.5,0.75,0.8,0.9,0.95])
for t in [60,90,120,180]:
    print(t, (behavioral_gaps<=t).mean()*100)
